---
### RAG PIPELINE
### DAta Ingestion to Vector DB Pipeline
---

In [10]:
### creating a funtion for reading documents and data from the folder

import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path


In [20]:
### read all the pdf inside the directory

def process_all_pdfs(pdf_directory):
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # find all the pdf files recursively in the directory
    pdf_files = list(pdf_dir.rglob("*.pdf"))
    print(f"Found {len(pdf_files)} PDF files in to process")


    for pdf_file in pdf_files:
        print(f"\nProcessing PDF file: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
                
            all_documents.extend(documents)
            print(f" ✅loaded {len(documents)} pages")


        except Exception as e:
            print(f" ❌ Error processing {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents


In [21]:
    ## Process all pdfs in the data directory
    all_pdf_documents = process_all_pdfs("../data/resume")

Found 15 PDF files in to process

Processing PDF file: Abhilash -Data Analyst - Resume.pdf
 ✅loaded 2 pages

Processing PDF file: Abhishak_Resume.pdf
 ✅loaded 1 pages

Processing PDF file: Aliakbar Kanorewala-Data Scientist Resume.pdf
 ✅loaded 2 pages

Processing PDF file: Chaitanya.M-FlowCV-Resume-20260204.pdf
 ✅loaded 1 pages

Processing PDF file: Data Scientist Resume 3.pdf
 ✅loaded 1 pages

Processing PDF file: Luis_Riera_Resume.pdf
 ✅loaded 1 pages

Processing PDF file: Michael_Okonji_Cloud_Engineer_Resume.pdf
 ✅loaded 3 pages

Processing PDF file: Prajakta_Koli_Data_Analyst.pdf
 ✅loaded 1 pages

Processing PDF file: Resume Suhas Patil_DA_BA.pdf
 ✅loaded 1 pages

Processing PDF file: resume_miroshkin_mts.pdf
 ✅loaded 2 pages

Processing PDF file: sudipto-baral-opensource.pdf
 ✅loaded 1 pages

Processing PDF file: Syed_Yaseen_DA_Resume_Gulf.pdf
 ✅loaded 2 pages

Processing PDF file: Updated Resume.pdf
 ✅loaded 2 pages

Processing PDF file: VIKAS REDDY_Cloud_Engineer.pdf
 ✅loaded 2 

In [22]:
all_pdf_documents

[Document(metadata={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2023-03-07T09:59:03+05:30', 'author': 'utsav mehta', 'moddate': '2023-03-07T09:59:03+05:30', 'source': '..\\data\\resume\\Abhilash -Data Analyst - Resume.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'Abhilash -Data Analyst - Resume.pdf', 'file_type': 'pdf'}, page_content='Abhilash B R                                    | +917892023766;  abhilash17br@gmail.com | https://www.linkedin.com/in/abhilash17br \n                                                                                           Portfolio Website:  https://abhilash17br.github.io      |     https://github.com/Abhilash17br \n \n"Data Enthusiast seeking opportunities in the field of Data Analytics to drive strategic business decisions through impactful    \ninsights and data-driven decision making.” \n \nPROJECT EXPERIENCE  \nBusiness Insights 360 - Data Analyst Challenge at Codebasics.io \nSQL 

In [26]:
###
### Text splitting get into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance."""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    ## show example of a chunk
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    ## show example of a chunk
    if split_docs:
        print(f"Example chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")  # Print first 200 characters
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs


In [27]:
## see the chunks
chunks=split_documents(all_pdf_documents)
chunks

Split 23 documents into 92 chunks
Example chunk:
Content: Abhilash B R                                    | +917892023766;  abhilash17br@gmail.com | https://www.linkedin.com/in/abhilash17br 
                                                                   ...
Metadata: {'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2023-03-07T09:59:03+05:30', 'author': 'utsav mehta', 'moddate': '2023-03-07T09:59:03+05:30', 'source': '..\\data\\resume\\Abhilash -Data Analyst - Resume.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'Abhilash -Data Analyst - Resume.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2023-03-07T09:59:03+05:30', 'author': 'utsav mehta', 'moddate': '2023-03-07T09:59:03+05:30', 'source': '..\\data\\resume\\Abhilash -Data Analyst - Resume.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'Abhilash -Data Analyst - Resume.pdf', 'file_type': 'pdf'}, page_content='Abhilash B R                                    | +917892023766;  abhilash17br@gmail.com | https://www.linkedin.com/in/abhilash17br \n                                                                                           Portfolio Website:  https://abhilash17br.github.io      |     https://github.com/Abhilash17br \n \n"Data Enthusiast seeking opportunities in the field of Data Analytics to drive strategic business decisions through impactful    \ninsights and data-driven decision making.” \n \nPROJECT EXPERIENCE  \nBusiness Insights 360 - Data Analyst Challenge at Codebasics.io \nSQL 

----
####
### EMBEDDINGS
####
### VECTORSTORE DB
####
----


In [ ]:
## For embedding we're going to update the requirements.txt file with some sentence transformers and hugging face libraries.
## additional libraries using : sentence-transformers , faiss-cpu , chromadb

In [28]:
# import some libraries
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

c:\Users\user\Downloads\demo-1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [37]:
## for embedding -- for managing the embedding 
class EmbeddingManager:
    """handles document embedding generation using sentence transformers"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the EmbeddingManager
        Args:
            model_name (str): HuggingFace model name for sentence embedding
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    
    def _load_model(self):
        """Load the sentence transformer model."""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"✅ model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"❌ Error loading model: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts.

        Args:
            texts (List[str]): List of text strings to embed
        
        return:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model is not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"✅ Generated embeddings with shape: {embeddings.shape}")
        return embeddings

# def get_embedding_dimension(self) -> int:
#       """Get the embedding dimension of the mode"""
#       if not self.model:
#           raise ValueError("Model not loaded")
#       return self.model.get_sentence_embedding_dimension()

## initialise the embedding manager
embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


c:\Users\user\Downloads\demo-1\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\user\Downloads\demo-1\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: U

✅ model loaded successfully. Embedding dimension: 384


C:\Users\user\AppData\Local\Temp\ipykernel_19780\1997154058.py:21: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"✅ model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


---
### vector store with persistant directory
---

In [48]:
## my chromadb client will be ready with my collection
class VectorStore:
    """Manages document embeddings in a chromadb vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialise the vector store

        Args:
            collection_name (str): Name of the collection in the vector store
            persist_directory (str): Directory to persist the vector store data
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize the ChromaDB client and collection."""
        try:
            # create persistent chromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            #get or create a collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "pdf document embeddings for RAG"}
            )
            print(f"Vector store initisalized. collection: {self.collection_name}")
            print(f"exisitng documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"❌ Error initializing vector store: {e}")
            raise

    ## when we create a collection, we need documents for it
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store.

        Args:
            documents (List[Any]): List of document objects (with metadata)
            embeddings (np.ndarray): Array of embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
    
        print(f"Adding {len(documents)} documents to vector store...")

        # prepare data for CHORMADB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}" ## creating uuid which is doc_id
            ids.append(doc_id)

            # prepare metadata
            metadata = dict(doc.metadata)  # ensure it's a dict
            metadata['doc_index'] = i  # add index for reference
            metadata['content_length'] = len(doc.page_content)  # add content length
            metadatas.append(metadata)

            #document content
            documents_text.append(doc.page_content)

            # embeddings
            embeddings_list.append(embedding.tolist())  # convert to list for ChromaDB

        # add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text,
            )
            print(f"✅ Successfully added {len(documents)} documents to vector store")
            print(f"total documents collection: {self.collection.count()}")

        except Exception as e:
            print(f"❌ Error adding documents to vector store: {e}")
            raise


#initialise vector store
vectorstore = VectorStore()
vectorstore


Vector store initisalized. collection: pdf_documents
exisitng documents in collection: 92


In [49]:
chunks

[Document(metadata={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2023-03-07T09:59:03+05:30', 'author': 'utsav mehta', 'moddate': '2023-03-07T09:59:03+05:30', 'source': '..\\data\\resume\\Abhilash -Data Analyst - Resume.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'Abhilash -Data Analyst - Resume.pdf', 'file_type': 'pdf'}, page_content='Abhilash B R                                    | +917892023766;  abhilash17br@gmail.com | https://www.linkedin.com/in/abhilash17br \n                                                                                           Portfolio Website:  https://abhilash17br.github.io      |     https://github.com/Abhilash17br \n \n"Data Enthusiast seeking opportunities in the field of Data Analytics to drive strategic business decisions through impactful    \ninsights and data-driven decision making.” \n \nPROJECT EXPERIENCE  \nBusiness Insights 360 - Data Analyst Challenge at Codebasics.io \nSQL 

In [50]:
### list comprehension
## convert text to embeddings

texts=[doc.page_content for doc in chunks]


## generate the embeddings
embeddings=embedding_manager.generate_embeddings(texts)

## store in the vector store
vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 92 texts...


Batches: 100%|██████████| 3/3 [00:03<00:00,  1.27s/it]

✅ Generated embeddings with shape: (92, 384)
Adding 92 documents to vector store...
✅ Successfully added 92 documents to vector store
total documents collection: 184


---
### RETRIEVER PIPELINE FROM VECTORSTORE
---

In [51]:
class RAGRetriever:
    """handles query based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the RAG Retriever.

        Args:
            vector_store (VectorStore): The vector store instance
            embedding_manager (EmbeddingManager): The embedding manager instance
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a given query.

        Args:
            query (str): The input query string
            top_k (int): Number of top documents to retrieve
            score_threshold: minimum similarity score threshold

        Returns:
            Liat of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score Threshold: {score_threshold}")

        #generate embedding for the query
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        #search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k,
            )   

            # proess results
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i,(doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score' : similarity_score,
                            'distance': distance,
                            'rank' : i+1
                        })

                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("no documents found")
            
            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vector_store,embedding_manager)

In [53]:
rag_retriever

In [58]:
## now testing the contents if able to find inside files
rag_retriever.retrieve("CloudWatch")

Retrieving documents for query: 'CloudWatch'
Top K: 5, Score Threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 54.82it/s]

✅ Generated embeddings with shape: (1, 384)
Retrieved 4 documents (after filtering)


[{'id': 'doc_3c91cc31_83',
  'content': 'ALB. \n• Monitoring infrastructure and applications using CloudWatch, CloudTrail, SNS, and AWS \nChatbot. \n• Used application logs and metrics from AWS CloudWatch to identify unusual patterns and \nsupport early issue detection. \n• Supported intelligent alerting workflows by structuring monitoring data to reduce noise and \nimprove incident response time. \n• Helped design scalable AWS infrastructure that can reliably support AI and data-driven \napplications by ensuring stable compute, storage, and networking.',
  'metadata': {'author': 'Jennifer Mbibe -X (jmbibe - INSIGHT GLOBAL INC at Cisco)',
   'file_type': 'pdf',
   'creationdate': '2026-01-29T11:41:58-05:00',
   'creator': 'Microsoft® Word for Microsoft 365',
   'page_label': '1',
   'moddate': '2026-01-29T11:41:58-05:00',
   'producer': 'Microsoft® Word for Microsoft 365',
   'content_length': 510,
   'doc_index': 83,
   'source_file': 'VIKAS REDDY_Cloud_Engineer.pdf',
   'total_pages'

---


- ## RETRIEVAL PIPELINE

---
### LLM
---
#### query retrieval pipeline {AUG + LLM}

---
#### integration of VECTORDB context pipeline with LLM Output


In [62]:
### simple RAG pipeline with Google-Gemini LLM
from langchain_google_genai import ChatGoogleGenerativeAI
import os
from dotenv import load_dotenv
load_dotenv()

## initialise the Gemini LLm with API Key
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

# Check API key
if not GOOGLE_API_KEY:
    raise ValueError("GOOGLE_API_KEY not found in .env file")

# Initialize Gemini LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-3-flash-preview",
    google_api_key=GOOGLE_API_KEY,
    temperature=0.1,
    max_output_tokens=1024
)

## 2. simple rag function : retrieve context + generate response
def    rag_simple(query,retriever,llm,top_k=3):
    ## retrieve relevant documents
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question"

## generate the answer using llm 
    prompt=f"""use the following context to answer the question concisely
        Context:
        {context}

        Question: {query}

        Answer:"""

    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content


In [ ]:
answer=rag_simple("Cloud Platforms & Services of VIKAS REDDY DEVARINTI?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'Cloud Platforms & Services of VIKAS REDDY DEVARINTI?'
Top K: 3, Score Threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 63.40it/s]

✅ Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)


[{'type': 'text', 'text': 'AWS (EC2, S3, ECS, EKS, ECR, IAM, RDS, VPC, Route 53, CloudWatch, CloudTrail, SNS), IBM Cloud, Oracle Cloud, Azure, and GCP.', 'extras': {'signature': 'EpsFCpgFAWkUfRO/zVXLT6fw2fi182MSAsgAjzHkbWOt5ZdwR+6PKAHNCMQ+WUs9MfzntYthtg3cMt1B8CW3/DQ3WynJmOoyz+bFi9QjEBbbOt+Yi9aeNGJNiYJo+gu3jvhymQyDqN8F5gpdJq5PNWSc/gcKeCpwzAD2mu/eGcILT+o+lkrgEoCYX/8SXED6Eaet2ayQqeJOy1Ycy5E6yLnmgVn53OxfKRStugtQd64XGh4krhG2MzI5rCpjs2HOCiAhxXAy1lynnUFd/kx5Cn5CEKoOMey3vJWL3sSbOqwAspONXkCfPlI7OdOBcfHKsb+tFvH93+ZTpiw5TcAmfL6ER5g8IS6tT78Wyp1s9EtitNTtU6xIN5x3EJdv3AbV5evYk36AhuB8QQA6xVpAgZ5itQqk+p0AUM3YF5g66sydDYns6SoV9Bxj6mZQAIWidqmYPxRtAxCuiuEl0szq9SPartUioP71104W6QtY7SL+Geh9bsnirl30MQlaggXsqvlDehaqPQqXS2LwuJJu/50uT1rDRWmjL6Nn0C22T79aH3BGDXrbGEjKvavhY/Pn6D6ZlwuYdslUATP6HJ+nVud3HhI0hwFEv3qRU4X2MZ4qtrcxNbtgo0PSCrkRg+v872v/2OZRFR0NY8/i4h9mKWhUgAEkgrf/uMdC8+RVxR6v7DufXo7ORmttZKFmk0EEJv5D9qxf9quHFMEOeMtZTh8VK8HecZqu7w0ypDLHY/7rjL1v2+Zx3wOvCt9+XLJ6cXMS4SWfwHtz2a4LLTSU3CqlvUnV8PkXNX62U2KNf/dnG4hyNZ2Hk

---
#### ENHANCED RAG Pipeline FEATURES
---


In [70]:
## ----------- enhanced rag pipeline features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG Pipeline with extra features:
    - return answers , sources, confidence score and optional full context
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}

    #prepare context and sources
    context = "\n\n".join([doc['context'] for doc in results])
    sources = [{ 
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')), 
        'page': doc['metadata'].get('page', 'unknown'), 
        'score': doc['similarity_score'], 
        'preview': doc['content'] [:120] + '...' 
    } for doc in results] 
    confidence = max([doc['similarity_score'] for doc in results])

## Generate answer
    prompt = f""" use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])

    output = {
        'answer' : response.content,
        'sources' : sources,
        'confidence' : confidence
    }
    if return_context:
        output['context'] = context
    return output


# Example usage: 
result = rag_advanced("Cloud Platforms & Services of VIKAS REDDY DEVARINTI?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True) 
print("Answer:", result['answer']) 
print("Sources:", result['sources']) 
print("Confidence:", result['confidence']) 
print("Context Preview:", result['context'] [:300])

Retrieving documents for query: 'Cloud Platforms & Services of VIKAS REDDY DEVARINTI?'
Top K: 3, Score Threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 77.59it/s]

✅ Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)
Answer: No relevant context found.
Sources: []
Confidence: 0.0
Context Preview: 
